# NL → FOL Pipeline — LogicLLaMA-style Prompt

**Dataset:** FOLIO (yale-nlp/folio)

**Prompt:** Adapted from LogicLLaMA / Harnessing the Power of LLMs (Yang et al., 2024)

**Models:** Configured in Cell 3 — swap `hf_path` to use larger models on the server.

---
## Prompt design note
The prompt in Table 5 of the paper is the **MALLS dataset generation prompt** used to create training data with GPT-4.
The inference prompt used to evaluate on FOLIO follows the same FOL notation rules but is structured
as a translation task with few-shot examples. This notebook implements that evaluation-style prompt,
using the same FOL operators, same predicate conventions, and same formatting rules from the paper.

## Cell 1 — Install Dependencies

In [1]:
%pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes datasets huggingface_hub pandas

## Cell 2 — HuggingFace Login

In [2]:
import os
from huggingface_hub import login

 

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Cell 3 — Configuration
**Only cell to modify on the server.**

In [3]:
MODEL_CONFIGS = [
    # {"name": "Llama-3.1-8B-Instruct", "hf_path": "meta-llama/Llama-3.1-8B-Instruct", "chat_template": "llama3"},
    # {"name": "Qwen2.5-7B-Instruct",   "hf_path": "Qwen/Qwen2.5-7B-Instruct",         "chat_template": "chatml"},
    {"name": "Qwen3-32B", "hf_path": "Qwen/Qwen3-32B", "chat_template": "qwen3"},
    # Server examples (uncomment and replace above):
    # {"name": "Llama-3.1-70B-Instruct", "hf_path": "meta-llama/Llama-3.1-70B-Instruct", "chat_template": "llama3"},
    # {"name": "Qwen2.5-72B-Instruct",   "hf_path": "Qwen/Qwen2.5-72B-Instruct",         "chat_template": "chatml"},
]
NUM_EXAMPLES   = 203        # increase to 203 for full FOLIO validation set
OUTPUT_JSON    = "fol_results_logicllama_prompt.json"
OUTPUT_CSV     = "fol_results_logicllama_prompt.csv"
MAX_NEW_TOKENS = 1024
TEMPERATURE    = 0.1
DO_SAMPLE      = True
USE_4BIT       = False     # set False on A100/H100

print("Config loaded:", [m["name"] for m in MODEL_CONFIGS])
print(f"Examples: {NUM_EXAMPLES} | 4-bit: {USE_4BIT}")

Config loaded: ['Qwen3-32B']
Examples: 203 | 4-bit: False


## Cell 4 — Load FOLIO Dataset

In [4]:
from datasets import load_dataset

def parse_premises(raw):
    if isinstance(raw, list):
        return [p.strip() for p in raw if p.strip()]
    return [p.strip() for p in raw.split("\n") if p.strip()]

folio    = load_dataset("yale-nlp/folio", split="validation")
examples = folio.select(range(NUM_EXAMPLES))
print(f"Loaded {len(folio)} total, using {NUM_EXAMPLES}")
ex = examples[0]
print("\nFirst example:")
for p in parse_premises(ex["premises"]): print(f"  - {p}")
print(f"Conclusion: {ex['conclusion']}")
print(f"Label: {ex['label']}")

Loaded 203 total, using 203

First example:
  - People in this club who perform in school talent shows often attend and are very engaged with school events.
  - People in this club either perform in school talent shows often or are inactive and disinterested community members.
  - People in this club who chaperone high school dances are not students who attend the school.
  - All people in this club who are inactive and disinterested members of their community chaperone high school dances.
  - All young children and teenagers in this club who wish to further their academic careers and educational opportunities are students who attend the school.
  - Bonnie is in this club and she either both attends and is very engaged with school events and is a student who attends the school or is not someone who both attends and is very engaged with school events and is not a student who attends the school.
Conclusion: Bonnie performs in school talent shows often.
Label: Uncertain


## Cell 5 — LogicLLaMA-style Prompt Builder

**Prompt design — based on Yang et al. (2024):**

The system prompt enforces the exact FOL operator set from the paper:
`⊕ ∨ ∧ → ∀ ∃ ¬ ↔` and explicitly forbids `=`, `!=`, `%` and bare variable expressions.
Predicate conventions follow the paper: predicates must always have a name and argument(s),
e.g. `Rounded(x)` not bare `x`. Quantifiers and variables go at the start of the formula.
Few-shot examples are drawn from the MALLS dataset examples shown in Table 5 of the paper.

In [5]:
# ── System prompt — enforces LogicLLaMA FOL notation rules ────────────
SYSTEM_PROMPT = """You are a logic expert. Translate natural language statements into \
first-order logic (FOL) rules.

FOL NOTATION RULES — follow exactly:
1. You MUST USE these logical operators: ⊕ (either or), ∨ (disjunction), ∧ (conjunction), \
   → (implication), ∀ (universal), ∃ (existential), ¬ (negation), ↔ (equivalence)
2. You MUST NEVER USE: =, !=, %, bare variable expressions like "y = a ∨ y = b"
3. Literals MUST always have a predicate name and entity, e.g. Rounded(x) or City(guilin)
4. ALWAYS put quantifiers and variables at the beginning of the FOL formula
5. Use only these variable sets: no variables; one variable x; two variables x,y; \
   three variables x,y,z
6. AVOID long predicate names — use short, clear names like Performs(x) not \
   PerformsInSchoolTalentShowOften(x)
7. The FOL MUST accurately reflect the meaning of the NL statement
"""

# ── Few-shot examples from MALLS / Table 5 of Yang et al. 2024 ────────
FEW_SHOT_EXAMPLES = """Examples of correct NL → FOL translation:

NL: If someone is entire, then he is not serious, and vice versa.
FOL: ∃x Entire(x) ↔ ¬Serious(x)

NL: If there is at least one person who is both not excited and not timid, then Jonathan is elderly.
FOL: (∃x ¬Excited(x) ∧ ¬Timid(x)) → Elderly(Jonathan)

NL: Someone who is either not fresh or entire is always not serious.
FOL: ∀x (¬Fresh(x) ∨ Entire(x)) → ¬Serious(x)

NL: If Nathalie is not blue, then Collier is entire.
FOL: ¬Blue(Nathalie) → Entire(Collier)

NL: Someone is courteous and not elderly if and only if he is not excited and not various.
FOL: ∃x (Courteous(x) ∧ ¬Elderly(x)) ↔ (¬Excited(x) ∧ ¬Various(x))
"""

def build_fol_prompt(premises, conclusion):
    pstr = "\n".join(f"{i+1}. {p}" for i, p in enumerate(premises))
    return (
        f"{FEW_SHOT_EXAMPLES}\n"
        "Now translate the following argument. "
        "Follow the FOL notation rules exactly.\n\n"
        "=== INPUT ===\n"
        f"PREMISES:\n{pstr}\n\n"
        f"CONCLUSION:\n{conclusion}\n\n"
        "=== OUTPUT FORMAT ===\n"
        "Respond using ONLY these four labeled sections:\n\n"
        "ENTITIES: entity1, entity2, ...\n\n"
        "PREDICATES:\n"
        "- PredicateName(x): short meaning\n\n"
        "PREMISES_FOL:\n"
        "1. <FOL formula>\n"
        "2. <FOL formula>\n\n"
        "CONCLUSION_FOL: <FOL formula>\n"
    )

# Preview
print(build_fol_prompt(parse_premises(examples[0]["premises"]), examples[0]["conclusion"]))

Examples of correct NL → FOL translation:

NL: If someone is entire, then he is not serious, and vice versa.
FOL: ∃x Entire(x) ↔ ¬Serious(x)

NL: If there is at least one person who is both not excited and not timid, then Jonathan is elderly.
FOL: (∃x ¬Excited(x) ∧ ¬Timid(x)) → Elderly(Jonathan)

NL: Someone who is either not fresh or entire is always not serious.
FOL: ∀x (¬Fresh(x) ∨ Entire(x)) → ¬Serious(x)

NL: If Nathalie is not blue, then Collier is entire.
FOL: ¬Blue(Nathalie) → Entire(Collier)

NL: Someone is courteous and not elderly if and only if he is not excited and not various.
FOL: ∃x (Courteous(x) ∧ ¬Elderly(x)) ↔ (¬Excited(x) ∧ ¬Various(x))

Now translate the following argument. Follow the FOL notation rules exactly.

=== INPUT ===
PREMISES:
1. People in this club who perform in school talent shows often attend and are very engaged with school events.
2. People in this club either perform in school talent shows often or are inactive and disinterested community members.


## Cell 6 — Output Parser

In [6]:
import re

def parse_fol_output(raw_text):
    result = {"entities": "", "predicates": "", "premises_fol": [],
              "conclusion_fol": "", "parse_success": False, "raw_output": raw_text.strip()}

    m = re.search(r"ENTITIES:\s*(.+?)(?:\n|$)", raw_text, re.IGNORECASE)
    if m: result["entities"] = m.group(1).strip()

    m = re.search(r"PREDICATES:\s*\n(.*?)(?=PREMISES_FOL:|STEP 3|$)", raw_text, re.IGNORECASE | re.DOTALL)
    if m: result["predicates"] = m.group(1).strip()

    m = re.search(r"PREMISES_FOL:\s*\n(.*?)(?=CONCLUSION_FOL:|STEP 4|$)", raw_text, re.IGNORECASE | re.DOTALL)
    if m:
        lines   = [l.strip() for l in m.group(1).strip().split("\n") if l.strip()]
        cleaned = [re.sub(r"^\d+[.)\s]+", "", l).strip() for l in lines]
        result["premises_fol"] = [c for c in cleaned if c]

    m = re.search(r"CONCLUSION_FOL:\s*(.+?)(?:\n|$)", raw_text, re.IGNORECASE)
    if m: result["conclusion_fol"] = m.group(1).strip()

    if result["premises_fol"] and result["conclusion_fol"]:
        result["parse_success"] = True

    if result["conclusion_fol"]:
        result["conclusion_fol"] = re.sub(r"^[\*\-\s]+", "", result["conclusion_fol"]).strip()

    return result

print("Parser ready.")

Parser ready.


## Cell 7 — Chat Template Formatter

In [7]:
def format_prompt(system, user, template, tokenizer=None):
    if template == "qwen3":
        if tokenizer is None:
            raise ValueError("Qwen3 formatting requires the loaded tokenizer.")
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    elif template == "llama3":
        return (
            "<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
        )
    elif template == "chatml":
        return (
            f"<|im_start|>system\n{system}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
    elif template == "gemma":
        return (
            f"<start_of_turn>user\n{system}\n\n{user}<end_of_turn>\n"
            "<start_of_turn>model\n"
        )
    else:
        raise ValueError(f"Unknown template: {template}")

print("Formatter ready.")

Formatter ready.


## Cell 8 — Model Loader / Runner

In [8]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

def load_model(hf_path):
    print(f"  Loading {hf_path} ...")
    tokenizer = AutoTokenizer.from_pretrained(hf_path, trust_remote_code=True)
    kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
        "torch_dtype": "auto",
    }
    if USE_4BIT:
        kwargs["quantization_config"] = bnb_config
    model = AutoModelForCausalLM.from_pretrained(hf_path, **kwargs)
    model.eval()
    print("  Model loaded.")
    return tokenizer, model

def unload_model(model, tokenizer):
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("  GPU memory freed.")

def generate_fol(model, tokenizer, prompt_text, max_new_tokens=512):
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(model.device)
    eos_id = tokenizer.eos_token_id
    if isinstance(eos_id, list):
        eos_id = eos_id[0]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=TEMPERATURE,
            do_sample=DO_SAMPLE,
            pad_token_id=eos_id,
            eos_token_id=eos_id,
            repetition_penalty=1.15,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Runner ready.")

Runner ready.


## Cell 9 — Main Pipeline Loop

In [9]:
all_results = []

for model_cfg in MODEL_CONFIGS:
    model_name, hf_path, chat_template = model_cfg["name"], model_cfg["hf_path"], model_cfg["chat_template"]
    print(f'\n{"="*60}\n MODEL: {model_name}\n{"="*60}')
    tokenizer, model = load_model(hf_path)

    for idx, example in enumerate(examples):
        premises   = parse_premises(example["premises"])
        conclusion = example["conclusion"]
        label      = example["label"]
        print(f"\n  [{idx+1}/{NUM_EXAMPLES}] Example {idx} ...")
        full_prompt = format_prompt(
            SYSTEM_PROMPT,
            build_fol_prompt(premises, conclusion),
            chat_template,
            tokenizer=tokenizer,
        )
        raw_output  = generate_fol(model, tokenizer, full_prompt, MAX_NEW_TOKENS)
        parsed      = parse_fol_output(raw_output)
        all_results.append({
            "model": model_name, "example_id": idx,
            "nl_premises": premises, "nl_conclusion": conclusion, "gold_label": label,
            "entities": parsed["entities"], "predicates": parsed["predicates"],
            "premises_fol": parsed["premises_fol"], "conclusion_fol": parsed["conclusion_fol"],
            "parse_success": parsed["parse_success"], "raw_output": parsed["raw_output"],
        })
        status = "OK" if parsed["parse_success"] else "FAILED"
        print(f"    Parse: {status} | Premises: {len(parsed['premises_fol'])} | Conclusion: {parsed['conclusion_fol'][:60]}")

    unload_model(model, tokenizer)

print(f"\nDone. Total results: {len(all_results)}")


 MODEL: Qwen3-32B
  Loading Qwen/Qwen3-32B ...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  Model loaded.

  [1/203] Example 0 ...
    Parse: OK | Premises: 6 | Conclusion: PerformOft(Bonnie)

  [2/203] Example 1 ...
    Parse: OK | Premises: 6 | Conclusion: (YongTeenAcadWish(Bonnie) ∧ ChaperoneDance(Bonnie)) ∨ (¬Youn

  [3/203] Example 2 ...
    Parse: OK | Premises: 6 | Conclusion: (ChaperoneDance(Bonnie) ∨ (¬ChaperoneDance(Bonnie) → Perform

  [4/203] Example 3 ...
    Parse: OK | Premises: 7 | Conclusion: HasLunchAtCompany(James)

  [5/203] Example 4 ...
    Parse: OK | Premises: 7 | Conclusion: ¬HaveLunchAtCompany(James)

  [6/203] Example 5 ...
    Parse: OK | Premises: 7 | Conclusion: (Manager(James) ∨ InOtherCountries(James)) → ¬(HaveLunchAtHo

  [7/203] Example 6 ...
    Parse: OK | Premises: 6 | Conclusion: ∃z IsAnimal(z)

  [8/203] Example 7 ...
    Parse: OK | Premises: 6 | Conclusion: ∀x ¬GetsFlu(x)

  [9/203] Example 8 ...
    Parse: OK | Premises: 6 | Conclusion: ∃s (SymptomOf(s, Monkeypox) ∧ s = Coughing)

  [10/203] Example 9 ...
    Parse: FAILED | Premise

## Cell 10 — Save Results

In [10]:
import json, pandas as pd

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"JSON saved: {OUTPUT_JSON}")

flat = [{
    "model": r["model"], "example_id": r["example_id"],
    "nl_premises": " | ".join(r["nl_premises"]), "nl_conclusion": r["nl_conclusion"],
    "gold_label": r["gold_label"], "entities": r["entities"],
    "premises_fol": " | ".join(r["premises_fol"]), "conclusion_fol": r["conclusion_fol"],
    "parse_success": r["parse_success"],
} for r in all_results]

df = pd.DataFrame(flat)
df.to_csv(OUTPUT_CSV, index=False)
print(f"CSV saved: {OUTPUT_CSV}")
print(f"Rows: {len(df)} | Models: {list(df['model'].unique())}")

# Uncomment for Google Drive:
# from google.colab import drive; drive.mount("/content/drive")
# df.to_csv("/content/drive/MyDrive/AGI_Research/fol_results_logicllama_prompt.csv", index=False)

JSON saved: fol_results_logicllama_prompt.json
CSV saved: fol_results_logicllama_prompt.csv
Rows: 203 | Models: ['Qwen3-32B']


## Cell 11 — Summary Report

In [11]:
print("\n" + "="*60 + "\n SUMMARY\n" + "="*60)
summary = df.groupby("model").agg(
    total=("example_id","count"), success=("parse_success","sum")
).reset_index()
summary["parse_rate"] = (summary["success"] / summary["total"] * 100).round(1).astype(str) + "%"
print(summary.to_string(index=False))

print("\n--- Sample (Example 0, first model) ---")
r = all_results[0]
print(f"Predicates:\n{r['predicates']}")
for i, p in enumerate(r["premises_fol"]): print(f"  {i+1}. {p}")
print(f"Conclusion: {r['conclusion_fol']}")


 SUMMARY
    model  total  success parse_rate
Qwen3-32B    203      202      99.5%

--- Sample (Example 0, first model) ---
Predicates:
- ClubMember(x): x is a member of the club  
- PerformOft(x): x performs in school talent shows often  
- Engaged(x): x attends and is very engaged with school events  
- InactiveDisint(x): x is an inactive and disinterested community member  
- ChaperoneDance(x): x chaperones high school dances  
- StudentAtSchl(x): x is a student who attends the school  
- WishAcadCareer(x): x wishes to further their academic career and educational opportunities
  1. ∀x (ClubMember(x) ∧ PerformOft(x)) → (Engaged(x))
  2. ∀x (ClubMember(x) → (PerformOft(x) ∨ InactiveDisint(x)))
  3. ∀x (ClubMember(x) ∧ ChaperoneDance(x)) → ¬StudentAtSchl(x)
  4. ∀x (ClubMember(x) ∧ InactiveDisint(x)) → ChaperoneDance(x)
  5. ∀x ((YoungTeen(x) ∧ ClubMember(x) ∧ WishAcadCareer(x)) → StudentAtSchl(x))
  6. ClubMember(Bonnie) ∧ [((Engaged(Bonnie) ∧ StudentAtSchl(Bonnie)) ∨ (¬(Engaged(Bon

## Cell 12 — Inspect Any Result

In [12]:
INSPECT_MODEL   = MODEL_CONFIGS[0]["name"]
INSPECT_EXAMPLE = 0

match = [r for r in all_results if r["model"] == INSPECT_MODEL and r["example_id"] == INSPECT_EXAMPLE]
if match:
    r = match[0]
    print(f"Model: {r['model']} | Example: {r['example_id']} | Parse: {'OK' if r['parse_success'] else 'FAILED'}")
    print("\n-- NL INPUT --")
    for i, p in enumerate(r["nl_premises"]): print(f"  {i+1}. {p}")
    print(f"  Conclusion : {r['nl_conclusion']}")
    print(f"  Gold label : {r['gold_label']}")
    print("\n-- FOL OUTPUT --")
    print(f"Entities   : {r['entities']}")
    print(f"Predicates :\n{r['predicates']}")
    for i, p in enumerate(r["premises_fol"]): print(f"  {i+1}. {p}")
    print(f"Conclusion : {r['conclusion_fol']}")
    print("\n-- RAW OUTPUT --")
    print(r["raw_output"])
else:
    print("No match found.")

Model: Qwen3-32B | Example: 0 | Parse: OK

-- NL INPUT --
  1. People in this club who perform in school talent shows often attend and are very engaged with school events.
  2. People in this club either perform in school talent shows often or are inactive and disinterested community members.
  3. People in this club who chaperone high school dances are not students who attend the school.
  4. All people in this club who are inactive and disinterested members of their community chaperone high school dances.
  5. All young children and teenagers in this club who wish to further their academic careers and educational opportunities are students who attend the school.
  6. Bonnie is in this club and she either both attends and is very engaged with school events and is a student who attends the school or is not someone who both attends and is very engaged with school events and is not a student who attends the school.
  Conclusion : Bonnie performs in school talent shows often.
  Gold label

In [18]:
import json
import pandas as pd

print("=" * 80)
print("CSV VALUES: fol_results_logicllama_prompt.csv")
print("=" * 80)
df = pd.read_csv("fol_results_logicllama_prompt.csv")
print(df.to_string(index=False))

print("\n" + "=" * 80)
print("JSON VALUES: fol_results_logicllama_prompt.json")
print("=" * 80)
with open("fol_results_logicllama_prompt.json", "r", encoding="utf-8") as f:
    json_values = json.load(f)
print(json.dumps(json_values, indent=2, ensure_ascii=False))

CSV VALUES: fol_results_logicllama_prompt.csv
    model  example_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      nl_premises                                                                                    